# PMU v3 pipeline — Kaggle

Single-cell launcher. All the heavy lifting lives in the `aciderix/Eurexplo`
repo under the branch `claude/review-pmu-scraper-nXJcB`. This notebook:

1. Clones the repo;
2. Installs `requirements_v3.txt`;
3. Restores previous checkpoints from any attached dataset;
4. Runs the resumable orchestrator with a wall-clock budget;
5. Snapshots everything under `/kaggle/working/outputs` so results survive.

See `kaggle_runner/README.md` in the repo for the full workflow and knobs.

In [ ]:
import os, sys, subprocess

# ── knobs ─────────────────────────────────────────────────────────────
os.environ.setdefault("PMU_BRANCH",            "claude/setup-kaggle-pmu-pipeline-DIXU0")
os.environ.setdefault("PMU_REPO_URL",          "https://github.com/aciderix/Eurexplo.git")
os.environ.setdefault("PMU_GPU",               "1")
os.environ.setdefault("PMU_BUDGET_MIN",        "480")
os.environ.setdefault("PMU_OPTUNA_TRIALS",     "80")
os.environ.setdefault("PMU_OPTUNA_SAMPLE_FRAC", "0.3")
os.environ.setdefault("PMU_STATE_DATASET",     "aciiderixx/pmu-v3-state")

# Auto-push state dataset needs Kaggle creds. On Kaggle notebooks add them in
# the left-pane "Add-ons → Secrets" as KAGGLE_USERNAME / KAGGLE_KEY; they then
# appear as env vars below.
for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
    if os.environ.get(k):
        import pathlib, json as _j, stat
        kdir = pathlib.Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
        (kdir / "kaggle.json").write_text(_j.dumps({
            "username": os.environ["KAGGLE_USERNAME"],
            "key":      os.environ["KAGGLE_KEY"]}))
        (kdir / "kaggle.json").chmod(0o600)
        break

repo = "/kaggle/working/repo"
if not os.path.exists(repo):
    subprocess.check_call(["git", "clone", "--depth=1", "--branch",
                           os.environ["PMU_BRANCH"], os.environ["PMU_REPO_URL"], repo])
else:
    subprocess.check_call(["git", "-C", repo, "fetch", "--depth=1", "origin",
                           os.environ["PMU_BRANCH"]])
    subprocess.check_call(["git", "-C", repo, "checkout", "FETCH_HEAD"])

sys.path.insert(0, repo)

subprocess.check_call([sys.executable, "-m", "kaggle_runner.kernel_entrypoint"],
                      cwd=repo)
